In [3]:
from mip import Model, MAXIMIZE, CBC, INTEGER, OptimizationStatus, xsum,BINARY
from itertools import permutations
import random
import pandas as pd
import networkx as nx
#find all apgs in 
def read_graphs_from_g6(graph_name):

    G_l = nx.read_graph6(graph_name)
    graphs = []
    intlist = random.sample(range(len(G_l)),10)
    #for ii in intlist:
    #for ii in range(200):
    for ii in range(len(G_l)):
    #for ii in range(100):
        g = G_l[ii]

        i = True
        print(ii)
        if ii != 0 and len(graphs) != 0:
            for g2 in graphs:
                if nx.is_isomorphic(g, g2) and len(graphs)!=1:
                    i = False
        if i:
            s = g.adjacency()
            listl = []
            for node, ad in s:
                l = []
                for i in range (g.number_of_nodes()):
                    if i in list(ad.keys()):
                        l.append(1)
                    else:
                        l.append(0)
                listl.append(l)
            #print(listl)
            graphs.append(g)

    return graphs

def pythonmip (G, a,d,col_name):

    print('fsdfds')
    n=G.order()
    m=G.size()

    const = a+(n-2)*d
    a_list = []
    list2=[]

    for i in range(n):
        a_list.append(a + i*d)
        list2.append(0)
    #print(a_dict)

    #assert (_sage_const_2 *n*aa+(n-_sage_const_1 )*n*dd)%_sage_const_4 ==_sage_const_0

    print('running:', a, d)
    for p in permutations(range(n)):
        #vertex labels
        B = [a+(p[i])*d for i in range(n)]
        p=Model(sense='MIN', solver_name='CBC')
        #b=p.add_var()
        b = {(u, v): p.add_var(var_type=INTEGER, name=f"b_{u}_{v}") for (u, v) in G.edges()}

        for i in range(n):
            #p += xsum(b[(u, v)] for (u, v) in G.edges() if i == u or i == v) >= B[i]
            #sum of all edge of vertex i
            edge_sum = xsum(b[(u, v)] for (u, v) in G.edges() if i == int(u) or i == int(v))
            #add constrains to model
            p += edge_sum == B[i]
            #p += xsum([b[(u,v)] for (u,v) in G.edges() \
                #if i==int(u) or i==int(v)]),min=B[i],max=B[i])
        p.objective = xsum(b[x] for x in G.edges())
        for e in G.edges():
            p.add_constr(b[e]>=1)


        # Solve the model
        p.verbose = 0
        p.optimize()


        # Check if a solution was found'''
        if p.status == OptimizationStatus.OPTIMAL:
            solution = {var.name: var.x for var in b.values()}
            # weights = {}
            for (edge, value) in solution.items():
                #print(edge)
                # vertex 1
                e1 = int(edge[2])
                # vertex 2
                e2 = int(edge[4])
                # set edge weight in G
                G[e1][e2]['weight'] = value
            print('uuu')
            ad_list = nx.adjacency_matrix(G, weight=None).toarray().tolist()
            weight = nx.get_edge_attributes(G, 'weight')
            return {col_name[0]: ad_list, col_name[1]: a, col_name[2]: d, col_name[3]: 'Y', col_name[4]: G,
                    col_name[5]: weight}
        elif p.status == OptimizationStatus.FEASIBLE:
            # Print the values of the variables if a solution exists
            #print('yes')
            solution = {var.name: var.x for var in b.values()}
            #weights = {}
            for (edge, value) in solution.items():
                #print(edge)
                #vertex 1
                e1 = int(edge[2])
                #vertex 2
                e2 = int(edge[4])
                #set edge weight in G
                G[e1][e2]['weight'] = value
                #weights[(e1,e2)] = value
                #store all vertex values in list2 by adding all its edges values
                list2[e1] += value
                list2[e2] += value
            #check if the vertex values illegal
            if sorted(a_list) == sorted(list2):
                print('uuu')
                ad_list = nx.adjacency_matrix(G, weight=None).toarray().tolist()
                weight = nx.get_edge_attributes(G, 'weight')
                return {col_name[0]: ad_list, col_name[1]: a, col_name[2]: d, col_name[3]:'Y', col_name[4]:G, col_name[5]:weight}


                # Exit the program
                #sys.exit(0)
            return 0
                    #return 0
def find_apgs (a_min, a_max, d_min, d_max, graphs, df,col_name):

    print('fsdfds')
    all = len(graphs)
    count = 0
    for G in graphs:
        count += 1
        print(count,'/',all)
        print('graph')
        for aa in range(a_min, a_max):
            for dd in range(d_min, d_max):
                print('ad')
                df2 = pythonmip(G, aa, dd, col_name)
                if df2 != 0 and df2 != None:
                    df = pd.concat([df, pd.DataFrame([df2])], ignore_index=True)
                    #df.to_csv('apgg8.csv')
    return df

def mip_find_apgs(graph_file_name, df_file_name):
    col_name = ['adjacency_list','a','d','APG_label_availability','graph','weight']
    dict = {col_name[0]:[],
            col_name[1]:[],
            col_name[2]:[],
            col_name[3]:[],
            col_name[4]:[],
            col_name[5]:[]
           }
    df = pd.DataFrame(dict)

    a_min = 2
    a_max = 4
    d_min = 1
    d_max = 4

    graphs = read_graphs_from_g6(graph_file_name)
    df = find_apgs(a_min, a_max, d_min, d_max, graphs, df, col_name)
    df.to_csv(df_file_name)

In [7]:
mip_find_apgs('/graphs_g6/graph3c.g6','/apgs/apg3.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/graphs_g6/graph3c.g6'